# Geracao de Parquets no Google Colab

Gera e valida os Parquets unificados por base a partir dos JSONLs ja existentes em `processed/textos_parlamentares/v1` no Google Drive. Este caderno nao executa a normalizacao nem depende de reabrir cadernos ja rodados.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

DATA_ROOT = Path('/content/drive/MyDrive/falando_nela/data')
os.environ['FALANDO_NELA_DATA_ROOT'] = str(DATA_ROOT)

PROCESSED_VERSION = 'v1'
PROCESSED_ROOT = DATA_ROOT / 'processed' / 'textos_parlamentares' / PROCESSED_VERSION
PARQUET_ROOT = PROCESSED_ROOT / 'parquet'
MANIFEST_ROOT = DATA_ROOT / 'processed' / 'manifests'

PARQUET_RUN_ID = f"processed-textos-v1-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-parquet"
OVERWRITE = True
BATCH_SIZE = 5000
VALIDATE_UNIQUE_IDS = True

print('DATA_ROOT:', DATA_ROOT)
print('PROCESSED_ROOT:', PROCESSED_ROOT)
print('PARQUET_ROOT:', PARQUET_ROOT)
print('PARQUET_RUN_ID:', PARQUET_RUN_ID)


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/pedblan/falando_nela.git'
REPO_DIR = Path('/content/falando_nela')
REPO_REF = ''  # opcional: branch, tag ou commit. Deixe vazio para usar o default remoto.

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags', '--prune'], check=True)
    if not REPO_REF:
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

if REPO_REF:
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', REPO_REF], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
print('Repo em:', Path.cwd())
subprocess.run(['git', 'status', '--short'], check=True)


## Conferir entrada

Confirma que os JSONLs processed ja estao no Drive e mostra os manifests recentes apenas como referencia. Manifests Parquet anteriores sao ignorados nesta lista.


In [ ]:
import json
from pathlib import Path

if not PROCESSED_ROOT.exists():
    raise FileNotFoundError(f'PROCESSED_ROOT nao encontrado: {PROCESSED_ROOT}')

jsonl_files = sorted(path for path in PROCESSED_ROOT.rglob('*.jsonl') if 'parquet' not in path.relative_to(PROCESSED_ROOT).parts)
processed_manifests = sorted(
    (path for path in MANIFEST_ROOT.glob('*.json') if not path.name.endswith('-parquet.json')),
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)
existing_parquets = sorted(PARQUET_ROOT.glob('*.parquet')) if PARQUET_ROOT.exists() else []

print('JSONLs processed encontrados:', len(jsonl_files))
print('Parquets existentes:', len(existing_parquets))
print('Manifests processed recentes:')
for path in processed_manifests[:10]:
    try:
        manifest = json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc:
        print(path.name, 'erro ao ler:', exc)
        continue
    print(path.name, manifest.get('run_id'), manifest.get('output_records'), manifest.get('output_record_counts'))


## Gerar Parquets

Lê todos os JSONLs processed v1 abaixo de `PROCESSED_ROOT`, deduplica por `texto_id` e grava um arquivo Parquet por `source/dataset` em `PARQUET_ROOT`.


In [ ]:
import json
from processamento.parquet import write_parquet_by_dataset

parquet_manifest = write_parquet_by_dataset(
    input_root=PROCESSED_ROOT,
    output_root=PARQUET_ROOT,
    manifest_path=MANIFEST_ROOT / f'{PARQUET_RUN_ID}.json',
    run_id=PARQUET_RUN_ID,
    overwrite=OVERWRITE,
    batch_size=BATCH_SIZE,
)

print(json.dumps({
    'manifest_path': parquet_manifest['manifest_path'],
    'output_root': parquet_manifest['output_root'],
    'input_file_count': parquet_manifest['input_file_count'],
    'output_file_count': parquet_manifest['output_file_count'],
    'output_records': parquet_manifest['output_records'],
    'output_record_counts': parquet_manifest['output_record_counts'],
    'skipped_counts': parquet_manifest['skipped_counts'],
}, ensure_ascii=False, indent=2))


## Validar saida

Confere se cada Parquet contem uma unica combinacao `source/dataset`, se `dataset_version` permanece `v1` e, quando habilitado, se nao ha `texto_id` duplicado dentro de cada arquivo.


In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

manifest_output_files = [Path(path) for path in parquet_manifest.get('output_files', [])]
if not manifest_output_files:
    raise RuntimeError('Nenhum Parquet foi gerado.')

row_total = 0
for path in manifest_output_files:
    if not path.exists():
        raise FileNotFoundError(path)
    table = pq.read_table(path, columns=['source', 'dataset', 'dataset_version', 'texto_id'])
    sources = set(table.column('source').to_pylist())
    datasets = set(table.column('dataset').to_pylist())
    versions = set(table.column('dataset_version').to_pylist())
    texto_ids = table.column('texto_id').to_pylist() if VALIDATE_UNIQUE_IDS else []
    unique_ids = len(texto_ids) == len(set(texto_ids)) if VALIDATE_UNIQUE_IDS else None
    if len(sources) != 1 or len(datasets) != 1:
        raise AssertionError(f'{path.name}: source/dataset misturados: {sources} {datasets}')
    if versions != {'v1'}:
        raise AssertionError(f'{path.name}: dataset_version inesperado: {versions}')
    if VALIDATE_UNIQUE_IDS and not unique_ids:
        raise AssertionError(f'{path.name}: texto_id duplicado')
    row_total += table.num_rows
    print({
        'arquivo': path.name,
        'linhas': table.num_rows,
        'source': next(iter(sources)),
        'dataset': next(iter(datasets)),
        'dataset_version': next(iter(versions)),
        'texto_id_unico': unique_ids,
    })

if row_total != parquet_manifest.get('output_records'):
    raise AssertionError(f'Total validado {row_total} difere do manifest {parquet_manifest.get("output_records")}')

print('Total validado:', row_total)
print('Manifest:', parquet_manifest['manifest_path'])
